In [ ]:
# Reader scale: frozen Kaggle stack and private HF credential.
import os, shutil, sys
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
from pathlib import Path
secret_value_0 = os.environ.get('HF_TOKEN')
if not secret_value_0:
    try:
        from kaggle_secrets import UserSecretsClient
        secret_value_0 = UserSecretsClient().get_secret('HF_TOKEN')
    except Exception:
        pass
pass
print('python', sys.version.split()[0], 'working disk', shutil.disk_usage('/kaggle/working'))
print('input mounts', sorted(str(p) for p in Path('/kaggle/input').iterdir()))


In [ ]:
# T4 dependencies
import subprocess, sys
try:
    _cap=subprocess.run(['nvidia-smi','--query-gpu=compute_cap','--format=csv,noheader'],capture_output=True,text=True,timeout=60).stdout.strip().splitlines()[0].strip()
except Exception: _cap=''
print('compute_cap:', _cap or 'unknown')
%pip install -q "transformers==5.17.0" datasets safetensors huggingface_hub matplotlib accelerate
import torch
assert torch.cuda.is_available(), 'need a GPU accelerator'
assert torch.cuda.get_device_capability(0) == (7, 5)
print('torch', torch.__version__, '| cap', torch.cuda.get_device_capability(0))
import transformers, datasets, safetensors, huggingface_hub
print(transformers.__version__, datasets.__version__, safetensors.__version__, huggingface_hub.__version__)

In [ ]:

from dataclasses import dataclass

@dataclass(frozen=True)
class Cfg:
    SOURCE_ID: str = 'Qwen/Qwen3.8-Flash-Next-FP8'
    TARGET_ID: str = 'Qwen/Qwen3.5-4B'
    MEM_DIM: int = 2560
    HIDDEN: int = 2560
    N_LAYERS: int = 32
    NGRAM: int = 3
    HEADS_PER_NGRAM: int = 8
    ROW_DIM: int = 160
    ROWS_PER_PART: int = 2_500_012
    VOCAB_BASE: int = 20_000_000
    SEED: int = 1234
    EOS: int = 248044
    VOCAB: int = 248320
    SEQ: int = 512
    VAL_FAST_TOKENS: int = 65536
    VAL_FULL_TOKENS: int = 524288
    DATASET_ID: str = 'HuggingFaceFW/fineweb-edu'
    DATASET_CONFIG: str = 'sample-10BT'
    SMOKE_TOKENS: int = 5120
    LR: float = 3e-5
    WD: float = 0.01
    WARMUP_FRAC: float = 0.05
    CKPTS: tuple = (100000, 250000, 500000)
    PLACEMENTS: tuple = ((3, 11),)
    BRANCHES: int = 1
    GAMMA_INIT: float = 1e-3
    SWEEP_ENABLED: bool = False
    BRANCH_ENABLED: bool = False
    REAL1M_ENABLED: bool = False
    REAL5M_ENABLED: bool = False
    STAGEA_ENABLED: bool = False

C = Cfg()
def describe(layers): return f"IDX {list(layers)} = HUMAN {[l+1 for l in layers]}"
print(C)
print('placements:', ' / '.join(describe(l) for l in C.PLACEMENTS))
for sites, R in [(2,1)]:
    n = sites*(R*C.MEM_DIM*C.HIDDEN + C.MEM_DIM*C.HIDDEN) + sites*R + sites
    print(f'sites={sites} R={R}: ~{n/1e6:.2f}M trainable')

SITES = (3, 11)
assert C.HIDDEN == 2560 and C.N_LAYERS == 32 and C.PLACEMENTS == (SITES,)

EXECUTION = {'strategy': 'single-T4-checkpointed', 'gpu': 0, 'decoder_gpu0': [0, 31], 'reader_devices': {'3': 0, '11': 0}, 'embedding_and_tied_head_gpu': 0, 'norm_gpu': 0, 'use_reentrant': False, 'preserve_rng_state': True, 'checkpoint_scope': 'decoder forward bodies only; module injection hooks execute once', 'reason': 'User requested single-GPU execution after ordinary single-T4 backward OOM', 'stock_logit_atol': 0.002, 'reader_gradient_atol': 1e-06, 'reader_gradient_rtol': 0.0001, 'reader_and_arbitration_math_unchanged': True, 'cuda_visible_devices': '0'}
GPU_IDS = (0,)

def gpu_sync():
    for i in GPU_IDS: torch.cuda.synchronize(i)

def gpu_reset():
    for i in GPU_IDS: torch.cuda.reset_peak_memory_stats(i)

def gpu_peaks():
    return [torch.cuda.max_memory_allocated(i)/1024**3 for i in GPU_IDS]


In [ ]:
# Cell 3 — EXACT source addressing (port of src/qwen36_ple/hashing.py — do not modify)
import math
import torch

MASK64=(1<<64)-1; GAMMA=0x9E3779B97F4A7C15; M1=0xBF58476D1CE4E5B9; M2=0x94D049BB133111EB; LPRIME=10007

def splitmix64(v):
    v=(v+GAMMA)&MASK64; v=((v^(v>>30))*M1)&MASK64; v=((v^(v>>27))*M2)&MASK64; return (v^(v>>31))&MASK64

def layer_multipliers(vocab, ngram, ple_idx=0, seed=1234):
    mmax=((1<<63)-1)//max(vocab,1); hb=max(1,mmax//2); base=seed+LPRIME*ple_idx
    return tuple(2*(splitmix64((base+GAMMA*(i+1))&MASK64)%hb)+1 for i in range(ngram))

def is_prime(v):
    if v<2: return False
    if v%2==0: return v==2
    return all(v%d for d in range(3, math.isqrt(v)+1, 2))

def nth_prime_after(s, k):
    p=s
    for _ in range(k):
        p+=1
        while not is_prime(p): p+=1
    return p

def head_layout(ngram=3, hpn=8, base=20_000_000, ple_idx=0):
    n=(ngram-1)*hpn
    sizes=tuple(nth_prime_after(base-1, ple_idx*n+h+1) for h in range(n))
    off=[]; t=0
    for s in sizes: off.append(t); t+=s
    return sizes, tuple(off)

def shift_right_ignore_eos(ids, shift, eos):
    if shift==0: return ids
    B,L=ids.shape; pos=torch.arange(L, device=ids.device)
    eos_pos=torch.where(ids==eos, pos, -1); prev_inc=torch.cummax(eos_pos,1).values
    prev=torch.cat([eos_pos.new_full((B,1),-1), prev_inc[:,:-1]],1)
    inseg=pos.unsqueeze(0)-(prev+1); src=pos-shift
    sh=ids.gather(1, src.clamp_min(0).unsqueeze(0).expand(B,-1))
    valid=(inseg>=shift)&(src.unsqueeze(0)>=0)
    return torch.where(valid, sh, ids.new_full((), eos))

def ngram_indices(ids, eos_token_id=248044, vocab_size=248320, ngram_size=3, heads_per_ngram=8,
                    vocab_size_base=20_000_000, ple_layer_index=0, seed=1234):
    ids=ids.long()
    mult=torch.tensor(layer_multipliers(vocab_size, ngram_size, ple_layer_index, seed), device=ids.device)
    sizes, offs=head_layout(ngram_size, heads_per_ngram, vocab_size_base, ple_layer_index)
    sizes=torch.tensor(sizes, device=ids.device); offs=torch.tensor(offs, device=ids.device)
    sh=[shift_right_ignore_eos(ids,s,eos_token_id) for s in range(ngram_size)]
    blocks=[]
    for ng in range(2, ngram_size+1):
        st=(ng-2)*heads_per_ngram; mixed=sh[0]*mult[0]
        for p in range(1,ng): mixed=torch.bitwise_xor(mixed, sh[p]*mult[p])
        blocks.append(torch.remainder(mixed.unsqueeze(-1), sizes[st:st+heads_per_ngram])+offs[st:st+heads_per_ngram])
    return torch.cat(blocks,-1)  # [B,L,16] GLOBAL PLE addresses, never raw token ids

_a=ngram_indices(torch.tensor([[1,2,3,4,5]])); _b=ngram_indices(torch.tensor([[1,2,3,4,5]]))
assert torch.equal(_a,_b) and _a.shape==(1,5,16)
SIZES, OFFS = head_layout()
print('hash ok; slots/head addrs e.g.', tuple(_a[0,2,:4].tolist()), '| head0 range', (OFFS[0], OFFS[0]+SIZES[0]))
def addresses(token_cpu):
    '''ONLY path from tokens to PLE rows: exact source hashes -> global head addresses.'''
    return ngram_indices(token_cpu, eos_token_id=C.EOS, vocab_size=C.VOCAB,
                         ngram_size=C.NGRAM, heads_per_ngram=C.HEADS_PER_NGRAM,
                         vocab_size_base=C.VOCAB_BASE, ple_layer_index=0, seed=C.SEED)

In [ ]:
# Cell 4 — shared-value reader (hidden=2560; reductions stay FP32 so fp16 backbone is safe)
import math
import torch
from torch import nn

def rms_norm(x, eps=1e-6):  # always FP32 reduction, cast back: fp16-safe
    return x.float().mul(torch.rsqrt(x.float().square().mean(-1, keepdim=True)+eps)).to(x.dtype)

class SharedValueReader(nn.Module):
    def __init__(self, mem_dim=2560, hidden=2560, branches=1, gamma=0.0):
        super().__init__(); self.mem_dim=mem_dim; self.hidden=hidden; self.branches=branches
        self.keys=nn.ModuleList(nn.Linear(mem_dim, hidden, bias=False) for _ in range(branches))
        self.value=nn.Linear(mem_dim, hidden, bias=False)
        self.beta=nn.Parameter(torch.zeros(branches))
        self.gamma=nn.Parameter(torch.tensor(float(gamma)))
        self.last_gate=None
    def stats(self):
        if self.last_gate is None: return None
        g=self.last_gate.float()
        return {'mean':g.mean().item(),'std':g.std(correction=0).item(),'near_zero':(g<0.01).float().mean().item()}
    def forward(self, h, m):
        assert h.shape[:-1]==m.shape[:-1], (h.shape, m.shape)
        h_dtype=h.dtype; h=h.float(); m=m.float()  # backbone may be fp16; reader computes FP32
        q=rms_norm(h); v=self.value(m); gs=[]
        for b,proj in enumerate(self.keys):
            k=rms_norm(proj(m))
            s=(q.float()*k.float()).sum(-1)/math.sqrt(self.hidden)
            gs.append(torch.sigmoid(s+self.beta[b].float()).to(v.dtype))
        g=torch.stack(gs,0)
        o=(g.unsqueeze(-1)*v.unsqueeze(0)).mean(0)
        self.last_gate=g.detach()
        return (h+self.gamma*o).to(h_dtype)  # residual back to backbone dtype; params stay FP32

_r=SharedValueReader(gamma=0.0); _h=torch.randn(1,4,2560); _m=torch.randn(1,4,2560)
assert torch.equal(_r(_h,_m),_h)
print('reader identity ok; R=1 params:', sum(p.numel() for p in _r.parameters()))

In [ ]:
# Cell 5 — injection hooks (IDX convention) + layer helper
import torch
from torch import nn

def decoder_layers(model):
    for path in ['model.layers','language_model.layers','transformer.h']:
        o=model
        try:
            for a in path.split('.'): o=getattr(o,a)
            if len(o)==32 or len(o)>0: return o
        except Exception: pass
    raise RuntimeError('decoder layers not found')

class ReaderInjection(nn.Module):
    '''layers = zero-based IDX list, exactly like the 35B run (e.g. (2,) = third block).'''
    def __init__(self, model, layers, mem_dim=2560, hidden=2560, branches=1, gamma=0.0):
        super().__init__()
        self.idx=tuple(layers)
        self.readers=nn.ModuleDict({str(l):SharedValueReader(mem_dim,hidden,branches,gamma) for l in layers})
        self.memory=None; self.handles=[]
        dec=decoder_layers(model)
        assert len(dec)==C.N_LAYERS, f'decoder count {len(dec)} != {C.N_LAYERS} — wrong hook target'
        for l in layers:
            self.handles.append(dec[l].register_forward_pre_hook(self._hook(str(l)), with_kwargs=True))
        print(f'inject at IDX {list(layers)} = HUMAN {[l+1 for l in layers]}')
    def _hook(self,name):
        def fn(mod,args,kw):
            if self.memory is None: return args,kw
            m=self.memory
            L=args[0].shape[1] if args else kw['hidden_states'].shape[1]
            if m.shape[1]!=L: m=m[:,:L]
            if args: return (self.readers[name](args[0],m),*args[1:]),kw
            kw['hidden_states']=self.readers[name](kw['hidden_states'],m); return args,kw
        return fn
    def set_memory(self,m): self.memory=m
    def close(self):
        [h.remove() for h in self.handles]; self.handles.clear()

print('injection ok')

In [ ]:
# Cell 6 — PLE stores: real (mount-only, exact scale or abort) + calibrated controls
import json, os
from pathlib import Path
import torch
from safetensors import safe_open

PLE_TMPL='model.language_model.layers.1.ple.ple_embedding.ngram_embedding.shard_{p}.weight'
PLE_SCALE='model.language_model.layers.1.ple.ple_embedding.ngram_embedding.weight_scale'

def rss_mb():
    '''Host RSS in MiB (Linux /proc; -1 if unavailable). Proves bounded RAM.'''
    try:
        with open('/proc/self/status') as _f:
            for _line in _f:
                if _line.startswith('VmRSS:'): return float(_line.split()[1])/1024
    except Exception: pass
    try:
        import resource; return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss/1024
    except Exception: return -1.0

def find_ple_manifests():
    hits=sorted(Path('/kaggle/input').rglob('manifest.json'))
    out=[]
    for h in hits:
        try:
            m=json.loads(h.read_text())
            if isinstance(m, dict) and 'parts' in m: out.append(h)
        except Exception: pass
    return out

class MountPLE:
    '''Real frozen PLE (row-level mmap, 35B-validated). REQUIRES /kaggle/input mount. Never downloads. Never caches parts: each lookup fetches ONLY needed rows via safetensors get_slice runs, so host RAM stays bounded after all 128 parts are touched.'''
    def __init__(self, manifest=None):
        manifests=[Path(manifest)] if manifest else find_ple_manifests()
        if not manifests or not all(p.exists() for p in manifests):
            raise RuntimeError('Real-PLE ABORT: no /kaggle/input PLE dataset attached. Attach the pinned shards+manifest.json dataset(s) first; refusing to download 48.7 GiB into /kaggle/working.')
        self.part_paths={}; revs=set()
        for mp in manifests:
            m=json.loads(mp.read_text())
            if not revs: self.rpp=m.get('rows_per_part',2500012); self.rd=m.get('row_dim',160)
            revs.add(m.get('ple_revision','unknown'))
            for k,v in m['parts'].items():
                p=mp.parent/v
                if not p.exists(): continue  # each dataset holds only its own shards
                if int(k) in self.part_paths:
                    assert self.part_paths[int(k)].name==p.name, f'part {k} filename clash'
                    continue
                self.part_paths[int(k)]=p
        assert len(revs)==1, f'mixed PLE revisions: {revs}'
        self.ple_revision=revs.pop()
        assert len(self.part_paths)==128, f'need all 128 parts, have {len(self.part_paths)}'
        missing=[str(p) for p in self.part_paths.values() if not p.exists()]
        if missing: raise RuntimeError(f"Real-PLE ABORT: {len(missing)} shard files missing, e.g. {missing[0]}")
        self.scale=self._resolve_scale()  # exact scale or abort — no fallback constant
        self.calls=0; self.rows_read=0
        self.parts_touched=set()  # cumulative DISTINCT parts; no part tensors ever held
        print(f'mounted PLE rev={self.ple_revision} parts={len(self.part_paths)} scale={self.scale}')
    def _resolve_scale(self):
        for f in sorted(set(self.part_paths.values())):
            try:
                with safe_open(f, framework='pt', device='cpu') as fh:
                    if PLE_SCALE in fh.keys():
                        return float(fh.get_tensor(PLE_SCALE).float().mean())
            except Exception: pass
        raise RuntimeError('Real-PLE ABORT: weight_scale tensor not found in pinned shards/index. Refusing hardcoded fallback.')
    def stats(self):
        return {'calls': self.calls, 'rows_read': self.rows_read,
                'parts_touched': len(self.parts_touched), 'held_part_tensors': 0,
                'scale': self.scale, 'rss_MiB': round(rss_mb(), 1)}
    @staticmethod
    def tensor_name(part): return PLE_TMPL.format(p=part)
    def lookup(self, indices):  # indices = GLOBAL head addresses [..,16] from ngram_indices()
        '''Row-level mmap reads: group deduped addresses by part (sorted), fetch ONLY
        needed rows as contiguous get_slice runs, dequantize the gathered rows. Full
        part tensors (~381 MiB each) are never materialized or cached.'''
        shape=indices.shape; flat=indices.detach().cpu().long().reshape(-1)
        uniq, inv=torch.unique(flat, return_inverse=True)  # dedup repeated addresses
        parts=torch.div(uniq, self.rpp, rounding_mode='floor'); local=uniq%self.rpp
        table=torch.empty(uniq.numel(), self.rd, dtype=torch.float32)
        for p in torch.unique(parts).tolist():  # ascending part order (page-friendly)
            pos=torch.nonzero(parts==p).flatten()
            lrows=local.index_select(0, pos); srows, sidx=torch.sort(lrows)
            runs=[]; a=int(srows[0]); prev=a
            for r in srows[1:].tolist():
                if r==prev+1: prev=r
                else: runs.append((a, prev+1)); a=prev=r
            runs.append((a, prev+1))
            path=self.part_paths.get(p)
            if path is None: raise FileNotFoundError(f'PLE part {p} not in manifest')
            with safe_open(str(path), framework='pt', device='cpu') as fh:
                sl=fh.get_slice(self.tensor_name(p))
                got=torch.cat([sl[x:y].to(torch.float32) for x, y in runs])*self.scale
            table.index_copy_(0, pos.index_select(0, sidx), got)  # got aligns with srows
        self.calls+=1; self.rows_read+=uniq.numel(); self.parts_touched.update(torch.unique(parts).tolist())
        return table[inv].reshape(*shape, self.rd).flatten(-2)  # [..,16,160]->[..,2560]

def permute_addresses(addrs, seed=777):
    '''Deterministic per-head bijective address permutation (shared by builder + store).'''
    sizes, offs=head_layout()
    S=torch.tensor(sizes); O=torch.tensor(offs)
    A=[]; B=[]
    for h,(s,o) in enumerate(zip(sizes, offs)):
        a=int(splitmix64((seed+10007*(h+1))&MASK64)%(s-1))+1
        b=int(splitmix64(((seed^0x9E3779B97F4A7C15)+7919*(h+1))&MASK64)%s)
        assert a%s!=0, 'A must be coprime to prime head size'
        A.append(a); B.append(b)
    A=torch.tensor(A); B=torch.tensor(B)
    f=addrs.long().cpu()
    H=f.shape[-1]
    O=O[:H]; A=A[:H]; B=B[:H]; S=S[:H]
    return O+((f-O)*A+B)%S

class RandomPLE:
    '''Calibrated per-head deterministic control: same global address -> same 160-d row, every call.
    Means/stds are per-head scalars measured from real rows in the frozen working set.
    16 rows concatenate to 2560-d. No 0.06 fallback.'''
    def __init__(self, head_means, head_stds, seed=0, row_dim=160):
        assert len(head_means)==16 and len(head_stds)==16, 'need 16 per-head stats'
        self.means=[float(m) for m in head_means]
        self.stds=[float(s) for s in head_stds]
        assert all(s>0 for s in self.stds), 'stds must be positive (calibrated)'
        self.seed=seed; self.rd=row_dim
        _sizes,_offs=head_layout()
        self._S=list(_sizes); self._O=list(_offs)
    def _head_of(self, a):
        for h in range(16):
            if self._O[h]<=a<self._O[h]+self._S[h]: return h
        raise ValueError('address outside head ranges')
    def _rows_for(self, uniq, heads):
        rows=[]
        for a,h in zip(uniq.tolist(), heads.tolist()):
            g=torch.Generator(); g.manual_seed((self.seed*1000003+int(a))%2**63)
            rows.append(torch.randn(self.rd, generator=g)*self.stds[h]+self.means[h])
        return torch.stack(rows)
    def lookup(self, indices):
        shape=indices.shape; flat=indices.detach().cpu().long().reshape(-1)
        uniq, inv=torch.unique(flat, return_inverse=True)
        heads=torch.tensor([self._head_of(int(a)) for a in uniq.tolist()])
        table=self._rows_for(uniq, heads)
        return table[inv].reshape(*shape, self.rd).flatten(-2)

def calibrate_head_stats(compact, n_per_head=4096, seed=0):
    '''Measure per-head mean/std from representative real rows in the compact working set.
    Samples n_per_head rows per head from the frozen prefix (train + full-val).'''
    sizes, offs=head_layout()
    addrs=compact.addrs.to(torch.int64)
    g=torch.Generator().manual_seed(seed)
    means=[]; stds=[]
    for h in range(16):
        lo=offs[h]; hi=offs[h]+sizes[h]
        pos=torch.nonzero((addrs>=lo)&(addrs<hi)).flatten()
        assert len(pos)>=n_per_head, f'head {h} only {len(pos)} rows'
        pick=pos[torch.randint(0,len(pos),(n_per_head,),generator=g)]
        rows=compact.rows.index_select(0, pick).to(torch.float32)*compact.scale
        means.append(float(rows.mean()))
        stds.append(float(rows.std(correction=0)))
    print('calibrated head means:', [round(m,6) for m in means])
    print('calibrated head stds:', [round(s,6) for s in stds])
    return means, stds

class PermutedPLE:
    '''Bijective per-head permutation: off_h + ((a-off_h)*A_h + B_h) % size_h.
    Preserves head ranges (sizes are prime, A_h % size_h != 0 so gcd=1 i.e. coprime) and table distribution.
    Wraps a compact working-set cache, never /kaggle/input random reads during training.'''
    def __init__(self, base, seed=777):
        self.b=base; self.seed=seed
        sizes, offs=head_layout()
        self.S=torch.tensor(sizes); self.O=torch.tensor(offs)
        A=[]; B=[]
        for h,(s,o) in enumerate(zip(sizes, offs)):
            a=int(splitmix64((seed+10007*(h+1))&MASK64)%(s-1))+1
            b=int(splitmix64(((seed^0x9E3779B97F4A7C15)+7919*(h+1))&MASK64)%s)
            assert a%s!=0, 'A must be coprime to prime head size'
            A.append(a); B.append(b)
        self.A=torch.tensor(A); self.B=torch.tensor(B)
    def lookup(self, indices):
        H=indices.shape[-1]; f=indices.long().cpu()
        O=self.O[:H]; A=self.A[:H]; B=self.B[:H]; S=self.S[:H]
        return self.b.lookup((O+((f-O)*A+B)%S).to(indices.device) if indices.is_cuda else (O+((f-O)*A+B)%S))

class CompactPLE:
    '''Compact working set for one frozen token prefix: sorted int32 addresses + fp8 rows (mmap).
    Bit-exact vs MountPLE: same bytes, same scale, same dequant formula. No 48.7 GiB traffic.'''
    def __init__(self, directory):
        import json as _json
        d=Path(directory)
        meta=_json.loads((d/'compact.json').read_text())
        n=meta['address_count']
        self.addrs=torch.from_file(str(d/'addrs.u32'), shared=True, size=n, dtype=torch.int32)
        raw=torch.from_file(str(d/'rows.u8'), shared=True, size=n*meta['row_dim'], dtype=torch.uint8)
        self.rows=raw.view(torch.float8_e4m3fn).view(n, meta['row_dim'])
        self.scale=float(meta['scale']); self.meta=meta
        self.ple_revision=meta.get('ple_revision')
        print('compact PLE: %d rows, %.2f GiB mapped, scale=%g' % (n, (d/'rows.u8').stat().st_size/1024**3, self.scale))
    def lookup(self, indices):
        shape=indices.shape; flat=indices.detach().cpu().long().reshape(-1)
        assert bool((flat>=0).all()) and int(flat.max())<2**31
        f32=flat.to(torch.int32)
        pos=torch.searchsorted(self.addrs, f32)
        posc=pos.clamp(max=len(self.addrs)-1)
        assert bool((self.addrs[posc]==f32).all()), 'compact miss: address outside frozen prefix'
        out=self.rows.index_select(0, posc).to(torch.float32)*self.scale
        return out.reshape(*shape, self.rows.shape[-1]).flatten(-2)



print('stores ok; manifests:', [str(p) for p in find_ple_manifests()])


In [ ]:
# Cell 7 — tokenizer verification: SEMANTIC id-space check (SPEC #15)
# Pinned-rev forensics: both model.vocab = 248044 entries with 0 id diffs; source-only
# ids are 7 audio added-tokens (248070-248076) above the target max: no collision.
# Config eos = 248044 (<|endoftext|>) on BOTH. AutoTokenizer.eos_token_id may report
# chat <|im_end|> 248046 instead: a chat-template default, NOT the PLE-training
# terminator. Hashing keeps training constants (vocab 248320 / eos 248044 / seed 1234).
import json
from transformers import AutoTokenizer
from huggingface_hub import HfApi, hf_hub_download

tok=secret_value_0
api=HfApi(token=tok)
trev='851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a'; srev='236dfdf285828023ca3bcd3f37366c58a3469b13'
print('target rev', trev[:12], '| source rev', srev[:12])
tt=AutoTokenizer.from_pretrained(C.TARGET_ID, token=tok, revision=trev)
st=AutoTokenizer.from_pretrained(C.SOURCE_ID, token=tok, revision=srev)
def raw_vocab(path):
    tj=json.load(open(path, encoding='utf-8'))
    m=dict(tj['model']['vocab'])
    for a in tj.get('added_tokens', []): m[a['content']]=a['id']
    return m
tm=raw_vocab(hf_hub_download(C.TARGET_ID,'tokenizer.json',revision=trev,token=tok))
sm=raw_vocab(hf_hub_download(C.SOURCE_ID,'tokenizer.json',revision=srev,token=tok))
diff={k for k in tm if k in sm and tm[k]!=sm[k]}
extra_t={k for k in tm if k not in sm}
smax=max(tm.values())
src_only={k: sm[k] for k in sm if k not in tm}
print('mapping diffs:', len(diff), '| target-only:', len(extra_t), '| source-only:', len(src_only))
assert not diff and not extra_t, 'target id space must match source ids exactly (native addressing)'
assert all(v>smax for v in src_only.values()), 'source-only ids must sit above target range'
assert sm.get('<|endoftext|>')==C.EOS and tm.get('<|endoftext|>')==C.EOS, 'training eos must be <|endoftext|> both sides'
probe='The quick brown fox 0123456789 function print(){} <|im_start|>x<|im_end|>'
assert tt.encode(probe, add_special_tokens=False)==st.encode(probe, add_special_tokens=False), 'probe encodings differ — STOP'
print('NATIVE token-ID addressing VALID: identical ids; training terminator eos =', C.EOS)


import hashlib
tp=hf_hub_download(C.TARGET_ID, 'tokenizer.json', revision=trev, token=tok)
sp=hf_hub_download(C.SOURCE_ID, 'tokenizer.json', revision=srev, token=tok)
assert hashlib.sha256(open(tp, 'rb').read()).hexdigest() == '5f9e4d4901a92b997e463c1f46055088b6cca5ca61a6522d1b9f64c4bb81cb42'
assert hashlib.sha256(open(sp, 'rb').read()).hexdigest() == '0997f410c57a1f4e53b09e4be8f4a172d90edd9564368fb0847030937229b9f3'
assert json.load(open(tp))['model']['merges'] == json.load(open(sp))['model']['merges']

tokenizer = tt


In [ ]:
# Cell 8 — immutable validation artifacts (built ONCE, never inside training) + FineWeb-Edu-only stream
import hashlib, json
from array import array
from pathlib import Path
from datasets import load_dataset

WORK=Path('/kaggle/working/qwengram-4b'); WORK.mkdir(parents=True, exist_ok=True)
VALDIR=WORK/'val-frozen-v1'; VALDIR.mkdir(exist_ok=True)

def _write_val(name, tokens_u32, meta_extra):
    tp=VALDIR/f'tokens-{name}.uint32le'; mp=VALDIR/f'validation-{name}.json'
    raw=tokens_u32.tobytes(); digest=hashlib.sha256(raw).hexdigest()
    meta={'name':name,'token_count':len(tokens_u32),'seq':512,'count':len(tokens_u32)//512,
            'tokens_sha256':digest, **meta_extra}
    if mp.exists():
        old=json.loads(mp.read_text())
        if old!=meta or tp.read_bytes()!=raw:
            raise RuntimeError(f'Immutable validation {name} differs — refusing to overwrite')
        print(f"reuse frozen val-{name} sha={digest[:16]} n={len(tokens_u32)}"); return meta
    tp.write_bytes(raw); mp.write_text(json.dumps(meta,indent=2)); print(f'wrote frozen val-{name} sha={digest[:16]}')
    return meta

def build_validation_artifacts(tok):
    '''Prefix-consistent: stream 524288 FineWeb-Edu tokens once; fast = first 65536 slice.'''
    paths = [VALDIR / ('validation-%s.json' % name) for name in ('fast', 'full')]
    assert all(p.exists() for p in paths), 'Frozen validation artifacts missing'
    return tuple(json.loads(p.read_text()) for p in paths)

def load_validation(name):
    m=json.loads((VALDIR/f'validation-{name}.json').read_text())
    raw=(VALDIR/f'tokens-{name}.uint32le').read_bytes()
    assert hashlib.sha256(raw).hexdigest()==m['tokens_sha256'], 'val checksum mismatch'
    a=array('I'); a.frombytes(raw)
    import torch
    return torch.tensor(a,dtype=torch.long).view(-1,512), m

class FineWebEduStream:
    '''FIRST experiment: FineWeb-Edu only (35B parity). No Stack/UltraChat/Cosmopedia yet.'''
    def __init__(self, tok, skip_docs, dataset_rev):
        '''dataset_rev MUST be the pinned rev from frozen validation metadata — never re-resolve latest.'''
        from datasets import load_dataset
        assert dataset_rev, 'pass dataset_rev from validation-full.json'
        self.ds=load_dataset(C.DATASET_ID, C.DATASET_CONFIG, split='train', streaming=True, revision=dataset_rev).skip(skip_docs)
        self.it=iter(self.ds)  # streaming datasets are not iterators themselves
        self.tok=tok
        self.documents_read=0
    def __iter__(self): return self
    def __next__(self):
        while True:
            t=(next(self.it) or {}).get('text') or ''
            self.documents_read+=1
            if str(t).strip(): return str(t)

def freeze_stream_tokens(tokenizer, stream, max_tokens, seq=512):
    '''EXACT token prefix train_reader consumes: same encode, same C.EOS, same chunking.
    Shared by the trainer and the compact-cache builder so both see identical tokens.'''
    buf = []
    out = []
    seen = 0
    while seen < max_tokens:
        while len(buf) < seq:
            buf += tokenizer.encode(next(stream), add_special_tokens=False) + [C.EOS]
        out.append(buf[:seq])
        buf = buf[seq:]
        seen += seq
    return torch.tensor(out, dtype=torch.long)

print('build with build_validation_artifacts(tok); read with load_validation("fast"/"full")')

In [ ]:
# Reuse and verify the exact frozen validation bytes.
import hashlib, shutil
_hits = list(Path('/kaggle/input').rglob('transfer-shas.json'))
assert len(_hits) == 1
_man = json.loads(_hits[0].read_text())
for _name in ('frozen-v1__tokens-fast.uint32le', 'frozen-v1__tokens-full.uint32le',
              'frozen-v1__validation-fast.json', 'frozen-v1__validation-full.json'):
    _src = _hits[0].parent / _name
    assert hashlib.sha256(_src.read_bytes()).hexdigest() == _man[_name]['sha256']
    _dst = VALDIR / _name.split('__', 1)[1]
    shutil.copyfile(_src, _dst)
    assert hashlib.sha256(_dst.read_bytes()).hexdigest() == _man[_name]['sha256']
# Cell 11b — validation prep: build immutable artifacts ONCE, then load both (idempotent)
mf, mfull = build_validation_artifacts(tokenizer)
val_fast, _ = load_validation("fast")
val_full, _ = load_validation("full")

print("validation ready")
print("fast:", mf["tokens_sha256"])
print("full:", mfull["tokens_sha256"])

from pathlib import Path
import sys
STAGE = _hits[0].parent
for name, record in _man.items():
    if name.startswith(('frozen-v1__', 'calib__')) and name.endswith(('.json', '.uint32le')):
        path = STAGE / name
        assert path.stat().st_size == record['size']
        assert hashlib.sha256(path.read_bytes()).hexdigest() == record['sha256']
for name in ('code__config.py', 'code__frozen_data.py'):
    path = STAGE / name
    assert hashlib.sha256(path.read_bytes()).hexdigest() == _man[name]['sha256']
    dest = WORK / name.split('__', 1)[1]
    shutil.copyfile(path, dest)
sys.path.insert(0, str(WORK))
import frozen_data
hellaswag, hs_meta = frozen_data.hellaswag_rows(1000, secret_value_0)
lambada, lam_meta = frozen_data.lambada_rows(1000, secret_value_0)
assert len(hellaswag) == len(lambada) == 1000
EVAL_MANIFEST = {'full': mfull['tokens_sha256'], 'fast': mf['tokens_sha256'],
                 'domains': {d: _man['frozen-v1__tokens-' + d + '.uint32le']['sha256']
                             for d in ('general', 'code', 'math', 'scientific', 'multilingual')},
                 'calibration': _man['calib__calib-750k-blocks.uint32le']['sha256'],
                 'hellaswag': hashlib.sha256(json.dumps(hellaswag, sort_keys=True).encode()).hexdigest(),
                 'lambada': hashlib.sha256(json.dumps(lambada, sort_keys=True).encode()).hexdigest(),
                 'hellaswag_source': hs_meta, 'lambada_source': lam_meta}
(WORK / 'evaluation-streams.json').write_text(json.dumps(EVAL_MANIFEST, indent=2))
print('frozen evaluation streams', json.dumps(EVAL_MANIFEST), flush=True)

assert EVAL_MANIFEST == {'full': '26ffe65b1f6457d2557dbacc98197d025932057e3c183e3cf1de8eefb7c2fe7c', 'fast': '4c702b4313fca2fa7fc5ffaf242912ab48a1564fecda52dd01982e2b756e4abd', 'domains': {'general': 'a9e12294eba002b5a4e9fce610ff39f8aed6d802e36e534e82ed367f96bb9aa5', 'code': '735f933ca4bc694562dbb2a6ab9c227c802268f7715448eba50e172ae450c421', 'math': 'c27f42f5f69de1aaf26020c7e14c2c141f8fa8490c5d95952c06e1c5bffa6648', 'scientific': 'c33632cf96170ca7ae92d4013702c14097fd79bb069c76e40fb3a531091936d8', 'multilingual': '65a26da5419aaa32520219d9d1c5909769a7a19a88ff16e85df7d69c38028a55'}, 'calibration': '379e484148e94b08e71870e6fabe67116260d965d9420b7a7d323cb09e4bd828', 'hellaswag': 'd4bc90defd0392ea41b0fa6e1e0cb31abec77af923abb8bc38f8c62100810123', 'lambada': '25852984eb52cef6a1af49308ecdea05c2e9a759a39c80fc64c51d58b4672028', 'hellaswag_source': {'dataset': 'Rowan/hellaswag', 'config': None, 'rev': '218ec52e09a7e7462a5400043bb9a69a41d06b76', 'n': 1000}, 'lambada_source': {'dataset': 'EleutherAI/lambada_openai', 'config': None, 'rev': '900124bf3b8235c6daf21033af9948b3f07346c4', 'n': 1000}}


In [ ]:
import hashlib, os, random, shutil, time
from array import array
from safetensors.torch import save_file, load_file

def sha(path):
    with Path(path).open('rb') as f:
        return hashlib.file_digest(f, 'sha256').hexdigest()

_train_hits = list(Path('/kaggle/input').rglob('reader-train-15m.uint32le'))
assert len(_train_hits) == 1
TRAIN_PATH = _train_hits[0]
TRAIN_MANIFEST_PATH = TRAIN_PATH.with_suffix('.json')
TRAIN_MANIFEST = json.loads(TRAIN_MANIFEST_PATH.read_text())
TRAIN_SHA = sha(TRAIN_PATH)
TRAIN_MANIFEST_SHA = sha(TRAIN_MANIFEST_PATH)
assert TRAIN_SHA == 'da94e4e3804753b25e4d05b6cef71975e7a943909302c5f3f5d81fddac9b04ef' == TRAIN_MANIFEST['sha256']
assert TRAIN_MANIFEST_SHA == 'a70009257b3662578f41d5dd0e51bf04f6338cc5b4fd0abe322904fdcbd85f2e'
assert TRAIN_MANIFEST['tokenizer_sha256'] == '5f9e4d4901a92b997e463c1f46055088b6cca5ca61a6522d1b9f64c4bb81cb42'
assert TRAIN_MANIFEST['token_count'] == 15000064
_tokens = array('I'); _tokens.frombytes(TRAIN_PATH.read_bytes())
FROZEN_TRAIN = torch.tensor(_tokens, dtype=torch.long).view(-1, 512)
assert FROZEN_TRAIN.shape == (29297, 512)
del _tokens
shutil.copyfile(TRAIN_MANIFEST_PATH, WORK / 'reader-train-15m.json')
print('Reused exact 2B frozen stream', TRAIN_SHA, TRAIN_MANIFEST_SHA, flush=True)


In [ ]:
# Cell 12e - COMPACT working set: exact sweep prefix + full validation, one sequential pass over /kaggle/input.
# /kaggle/input stays the immutable master. Cache lives in /kaggle/tmp (WORK fallback). One cache serves IDX 2/8/2+8.
try:
    _cstore=MountPLE()
except RuntimeError as _e:
    print('COMPACT SKIPPED (attach PLE datasets first):', str(_e)[:200]); _cstore=None
if _cstore is not None:
    import time, hashlib, json as _json
    from array import array as _array
    _t0=time.perf_counter()
    _mfF=json.loads((VALDIR/'validation-full.json').read_text())
    _train_toks=FROZEN_TRAIN[5000192//C.SEQ:10000384//C.SEQ]
    _train_sha=hashlib.sha256(_train_toks.numpy().astype('<u4').tobytes()).hexdigest()
    _val,_=load_validation('full')
    _all=torch.cat([_val, _train_toks])
    _addrs=ngram_indices(_all)
    _uniq=torch.unique(_addrs.reshape(-1))
    _totreq=int(_addrs.numel()); _nunq=len(_uniq)
    print('requested addresses: %d; unique: %d; dedup ratio %.2fx' % (_totreq, _nunq, _totreq/_nunq), flush=True)
    _cdir=Path('/kaggle/tmp/compact-reader-5-10m') if Path('/kaggle/tmp').exists() else WORK/'compact-reader-5-10m'
    _cdir.mkdir(parents=True, exist_ok=True)
    print('cache dir:', _cdir, flush=True)
    _af=(_cdir/'addrs.u32').open('wb'); _rf=(_cdir/'rows.u8').open('wb')
    _ha=hashlib.sha256(); _hr=hashlib.sha256(); _n=0; _prev=-1; _readb=0; _ts=time.perf_counter()
    _parts=torch.div(_uniq, C.ROWS_PER_PART, rounding_mode='floor'); _local=_uniq % C.ROWS_PER_PART
    _byfile={}
    for _p in torch.unique(_parts).tolist():
        _byfile.setdefault(str(_cstore.part_paths[_p]), []).append(_p)
    from safetensors import safe_open as _so
    for _fpath in sorted(_byfile):
        with _so(_fpath, framework='pt', device='cpu') as _fh:
            for _p in sorted(_byfile[_fpath]):
                _full=_fh.get_slice(MountPLE.tensor_name(_p))[:]  # one sequential full-part read
                _readb+=_full.numel()
                _pos=torch.nonzero(_parts==_p).flatten()
                _rows=_full.index_select(0, _local.index_select(0, _pos))
                _au=_uniq.index_select(0, _pos)
                assert int(_au[0])>_prev; _prev=int(_au[-1])
                _ab=_array('I', _au.tolist()).tobytes(); _rb=_rows.view(torch.uint8).contiguous().numpy().tobytes()
                _af.write(_ab); _rf.write(_rb); _ha.update(_ab); _hr.update(_rb); _n+=_pos.numel()
    _af.close(); _rf.close()
    _scant=_ts and (time.perf_counter()-_ts)
    _meta={'format':'qwen-ple-compact','version':1,'token_train':int(_train_toks.numel()),'token_val':int(_val.numel()),
           'requested_addresses':_totreq,'address_count':_n,'dedup_ratio':_totreq/_n,
           'row_dim':C.ROW_DIM,'scale':float(_cstore.scale),'ple_revision':_cstore.ple_revision,
           'dataset_rev':_mfF['dataset_rev'],'train_skip_docs':_mfF['skip_docs'],
           'full_train_sha256':TRAIN_SHA,'train_manifest_sha256':TRAIN_MANIFEST_SHA,
           'train_start':5000192,'train_end':10000384,'train_tokens_sha256':_train_sha,
           'target_rev':'851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a','tokenizer':'native token ids (Cell 7 semantic proof)',
           'hash':{'ngram':C.NGRAM,'heads_per_ngram':C.HEADS_PER_NGRAM,'vocab_size':C.VOCAB,'vocab_base':C.VOCAB_BASE,'seed':C.SEED,'eos':C.EOS},
           'addrs_sha256':_ha.hexdigest(),'rows_sha256':_hr.hexdigest(),'pattern':'freeze_stream_tokens+ngram_indices'}
    (_cdir/'compact.json').write_text(_json.dumps(_meta,indent=2))
    _dt=time.perf_counter()-_t0; _sz=(_cdir/'rows.u8').stat().st_size+(_cdir/'addrs.u32').stat().st_size
    print('COMPACT: %d unique rows (dedup %.2fx), %.3f GiB, built in %.0fs, source read %.0f MiB/s' % (_n,_totreq/_n,_sz/1024**3,_dt,_readb/_scant/1024**2), flush=True)
    _cmp=CompactPLE(_cdir)
    def _file_sha(path):
        h=hashlib.sha256()
        with open(path,'rb') as f:
            for block in iter(lambda:f.read(1<<20),b''): h.update(block)
        return h.hexdigest()
    assert _cmp.meta==_meta
    assert (_cdir/'addrs.u32').stat().st_size==_n*4
    assert (_cdir/'rows.u8').stat().st_size==_n*C.ROW_DIM
    assert _file_sha(_cdir/'addrs.u32')==_meta['addrs_sha256']
    assert _file_sha(_cdir/'rows.u8')==_meta['rows_sha256']
    assert bool((_cmp.addrs[1:]>_cmp.addrs[:-1]).all())
    print('compact cache SHA/reopen verified', _meta['train_tokens_sha256'], flush=True)
    _g=torch.Generator().manual_seed(0)
    _samp=_uniq[torch.randint(0,len(_uniq),(2048,),generator=_g)].reshape(128,16)
    _d=(_cmp.lookup(_samp)-_cstore.lookup(_samp)).abs().max().item()
    print('equivalence max|diff| on 2048 sampled addresses:', _d); assert _d==0.0
    print('COMPACT BUILD OK (same frozen addressing at IDX3 and IDX11)')

# Release construction tensors; CompactPLE keeps only the mapped cache.
import gc
for _name in ('_train_toks', '_all', '_addrs', '_uniq', '_parts', '_local', '_rows', '_full'):
    globals().pop(_name, None)
gc.collect()

assert _cstore is not None and _cmp is not None
CACHE_META = json.loads((_cdir / 'compact.json').read_text())
CACHE_SHA = CACHE_META['rows_sha256']
ADDR_SHA = CACHE_META['addrs_sha256']
assert CACHE_META['ple_revision'] == '236dfdf285828023ca3bcd3f37366c58a3469b13'
shutil.copyfile(_cdir / 'compact.json', WORK / 'compact-reader-5-10m.json')
print('cache row SHA', CACHE_SHA, 'mapping SHA', ADDR_SHA, flush=True)

assert CACHE_SHA == '337b79203d5f74436b23470964c68154ab6cc73ed38f7fc8b3fa30ea0fd9da61' and ADDR_SHA == '350908df637fd8fa4c62ddeed09dcfada54eec4fbf2ed3d2a2f31c18aa2639d7'


In [ ]:
# Cell 9 — frozen Qwen3.5-4B via full-config VLM-compat CausalLM (vision frozen, text path): float16 FIRST
import os, torch
from transformers import AutoConfig, AutoModelForCausalLM

tok=secret_value_0
from huggingface_hub import HfApi
trev='851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a'
cfg=AutoConfig.from_pretrained(C.TARGET_ID, token=tok, revision=trev, trust_remote_code=True)
assert getattr(cfg,'model_type',None)=='qwen3_5', getattr(cfg,'model_type',None)
tconf=cfg.text_config  # dims ONLY — never pass as config= (strips auto_map, breaks class resolution)
print('hidden',tconf.hidden_size,'layers',tconf.num_hidden_layers,'vocab',tconf.vocab_size,'arch',type(cfg).__name__)
assert tconf.hidden_size==2560 and tconf.num_hidden_layers==32
from transformers import AutoTokenizer
tokenizer=AutoTokenizer.from_pretrained(C.TARGET_ID, token=tok, revision=trev, trust_remote_code=True)

try:
    from transformers.models.qwen3_5 import Qwen3_5ForCausalLM as _Q
    print('direct qwen3_5 import ok:', _Q.__name__)
except Exception:
    import traceback; traceback.print_exc()
    raise RuntimeError('qwen3_5 modeling import failed — true cause above')
ACTIVE_DTYPE=None; model=None
for dt in [torch.float16]:
    try:
        m=AutoModelForCausalLM.from_pretrained(C.TARGET_ID, revision=trev, token=tok,
            trust_remote_code=True, device_map={'':0}, torch_dtype=dt, low_cpu_mem_usage=True)
        m.eval(); m.requires_grad_(False)
        assert sum(1 for p in m.parameters() if p.requires_grad)==0
        ids=tokenizer('The quick brown fox jumps over the lazy dog. '*8, return_tensors='pt').input_ids[:,:64].cuda()
        with torch.inference_mode():
            lg=m(input_ids=ids,use_cache=False).logits
        assert torch.isfinite(lg.float()).all(), 'non-finite logits'
        model=m; ACTIVE_DTYPE=dt; print(f'frozen load OK in {dt} footprint={m.get_memory_footprint()/1024**3:.2f} GiB')
        print('model class:', type(m).__name__, '| decoder blocks:', len(decoder_layers(m)))
        del ids, lg; break
    except Exception as e:
        print(f'{dt} rejected: {str(e)[:200]}')
        try: del m
        except Exception: pass
assert model is not None and ACTIVE_DTYPE is not None
print('backbone = ACTIVE_DTYPE:', ACTIVE_DTYPE)
print('reader params/compute = FP32 (AdamW FP32; norms/scores FP32)')
print('PLE dequant/output = FP32')
print('reader residual output is cast back to backbone dtype')

In [ ]:
import torch
from torch.utils.checkpoint import checkpoint


class CheckpointedDecoder:
    def __init__(self, layers):
        self.enabled = True
        self.checkpoint_calls = 0
        self.body_calls = 0
        self.originals = []
        for layer in layers:
            assert not getattr(layer, 'gradient_checkpointing', False)
            assert all(not p.requires_grad for p in layer.parameters())
            self.originals.append(layer.forward)
            layer.forward = self.wrap(layer.forward)

    def wrap(self, original):
        def body(*args, **kwargs):
            self.body_calls += 1
            return original(*args, **kwargs)

        def forward(hidden_states, *args, **kwargs):
            assert kwargs.get('past_key_values') is None
            if self.enabled and torch.is_grad_enabled() and hidden_states.requires_grad:
                self.checkpoint_calls += 1
                # Module hooks run outside forward, so injections are not replayed.
                return checkpoint(body, hidden_states, *args, use_reentrant=False,
                                  preserve_rng_state=True, **kwargs)
            return original(hidden_states, *args, **kwargs)
        return forward

    def description(self):
        return {'strategy':'single-T4-checkpointed', 'gpu':0,
                'decoder_layers':len(self.originals), 'use_reentrant':False,
                'preserve_rng_state':True, 'injection_hooks_recomputed':False,
                'backbone_eval_mode_preserved':True,
                'checkpoint_calls':self.checkpoint_calls, 'body_calls':self.body_calls}

import gc
_mp_ids = val_full[:1,:64].cuda()
_mp_memory = _cmp.lookup(addresses(_mp_ids.cpu())).cuda()
with torch.inference_mode():
    _mp_stock = model(input_ids=_mp_ids, use_cache=False).logits.detach().cpu()
torch.manual_seed(1234)
_mp_probe = ReaderInjection(model,SITES,C.MEM_DIM,C.HIDDEN,1,C.GAMMA_INIT).cuda()
_mp_probe.set_memory(_mp_memory)
_mp_state = {k:v.detach().cpu().clone() for k,v in _mp_probe.state_dict().items()}
_mp_logits = model(input_ids=_mp_ids,use_cache=False).logits
torch.nn.functional.cross_entropy(_mp_logits[:,:-1].float().reshape(-1,C.VOCAB),_mp_ids[:,1:].reshape(-1)).backward()
_mp_expected = _mp_logits.detach().cpu()
_mp_grads = {k:p.grad.detach().cpu().clone() for k,p in _mp_probe.named_parameters()}
_mp_probe.close()
del _mp_probe,_mp_logits
gc.collect(); torch.cuda.empty_cache()
EXECUTOR = CheckpointedDecoder(model.model.layers)
with torch.inference_mode():
    _mp_actual_stock = model(input_ids=_mp_ids,use_cache=False).logits.detach().cpu()
_mp_stock_diff = (_mp_stock.float()-_mp_actual_stock.float()).abs().max().item()
assert _mp_stock_diff <= 0.002
_mp_probe = ReaderInjection(model,SITES,C.MEM_DIM,C.HIDDEN,1,C.GAMMA_INIT).cuda()
_mp_probe.load_state_dict(_mp_state); _mp_probe.set_memory(_mp_memory)
_mp_logits = model(input_ids=_mp_ids,use_cache=False).logits
torch.nn.functional.cross_entropy(_mp_logits[:,:-1].float().reshape(-1,C.VOCAB),_mp_ids[:,1:].reshape(-1)).backward()
_mp_reader_diff = (_mp_expected.float()-_mp_logits.detach().float().cpu()).abs().max().item()
assert _mp_reader_diff <= 0.002
_mp_grad_diff = {}
for name,p in _mp_probe.named_parameters():
    torch.testing.assert_close(p.grad.cpu(),_mp_grads[name],atol=1e-6,rtol=1e-4)
    _mp_grad_diff[name] = (p.grad.cpu()-_mp_grads[name]).abs().max().item()
assert all(not p.requires_grad and p.grad is None for p in model.parameters())
if EXECUTION['strategy'] == 'single-T4-checkpointed':
    assert all(p.device.index == 0 for p in model.parameters())
    assert EXECUTOR.body_calls > EXECUTOR.checkpoint_calls > 0
_mp_probe.close()
MP_PARITY = {**EXECUTOR.description(),'stock_max_abs_logit':_mp_stock_diff,
             'reader_max_abs_logit':_mp_reader_diff,'reader_gradient_max_abs':_mp_grad_diff,
             'logit_atol':0.002,'gradient_atol':1e-6,'gradient_rtol':1e-4}
(WORK/'execution-parity.json').write_text(json.dumps(MP_PARITY,indent=2))
print('EXECUTION_PARITY',json.dumps(MP_PARITY),flush=True)
del p,_mp_probe,_mp_state,_mp_grads,_mp_logits,_mp_ids,_mp_memory,_mp_expected,_mp_stock,_mp_actual_stock
if hasattr(EXECUTOR,'cache'): EXECUTOR.cache.clear()
gc.collect()
for i in GPU_IDS:
    with torch.cuda.device(i): torch.cuda.empty_cache()


In [ ]:
import gc, json, time, platform, importlib.metadata
import torch.nn.functional as F

RUNTIME = {'python':platform.python_version(), 'torch':torch.__version__,
           'cuda_visible_devices':os.environ.get('CUDA_VISIBLE_DEVICES'),
           'cuda':torch.version.cuda, 'cudnn':torch.backends.cudnn.version(),
           'packages':{k:importlib.metadata.version(k) for k in ('transformers','safetensors','accelerate','tokenizers')},
           'gpus':[{'name':torch.cuda.get_device_name(i),
                    'total_bytes':torch.cuda.get_device_properties(i).total_memory,
                    'capability':list(torch.cuda.get_device_capability(i))} for i in range(torch.cuda.device_count())]}
(WORK / 'runtime.json').write_text(json.dumps(RUNTIME, indent=2))
assert C.HIDDEN == 2560 and C.N_LAYERS == 32 and SITES == (3, 11)
assert all(not p.requires_grad and p.grad is None for p in model.parameters())
assert torch.cuda.get_device_capability(0) == (7, 5)
if EXECUTION['strategy'] == 'single-T4-checkpointed': assert torch.cuda.device_count() == 1
assert ACTIVE_DTYPE == torch.float16
assert all(next(decoder_layers(model)[i].parameters()).device.index == 0 for i in SITES)
assert all(decoder_layers(model)[i].block_type == 'full_attention' for i in SITES)
assert _cstore.ple_revision == '236dfdf285828023ca3bcd3f37366c58a3469b13'
assert all(Path(p).is_relative_to('/kaggle/input') for p in _cstore.part_paths.values())
PLE_STATS = {str(p): (p.stat().st_size, p.stat().st_mtime_ns) for p in _cstore.part_paths.values()}
BACKBONE_VERSIONS = {n: p._version for n, p in model.named_parameters()}
_qual_start = int(_cmp.meta['train_start']) // C.SEQ
_qual_ids = FROZEN_TRAIN[_qual_start:_qual_start+1, :64].to('cuda')
_qual_mem = _cmp.lookup(addresses(_qual_ids.cpu())).to('cuda')
with torch.inference_mode():
    _stock = model(input_ids=_qual_ids, use_cache=False).logits
_probe = ReaderInjection(model, SITES, C.MEM_DIM, C.HIDDEN, 1, 0.0).to('cuda')
assert _probe.idx == SITES
with torch.inference_mode():
    _probe.set_memory(None)
    _disabled = model(input_ids=_qual_ids, use_cache=False).logits
    _disabled_diff = (_stock.float() - _disabled.float()).abs().max().item()
    _probe.set_memory(_qual_mem)
    _zero = model(input_ids=_qual_ids, use_cache=False).logits
    _zero_diff = (_stock.float() - _zero.float()).abs().max().item()
assert _disabled_diff <= 0.002 and _zero_diff <= 0.002
with torch.no_grad():
    for reader in _probe.readers.values(): reader.gamma.fill_(C.GAMMA_INIT)
    _near = model(input_ids=_qual_ids, use_cache=False).logits
assert torch.isfinite(_near).all()
_near_diff = (_stock.float() - _near.float()).abs().max().item()
_params = list(_probe.parameters())
assert len(_params) == 8 and all(p.dtype == torch.float32 for p in _params)
_opt = torch.optim.AdamW(_params, lr=C.LR, weight_decay=C.WD)
assert {id(p) for g in _opt.param_groups for p in g['params']} == {id(p) for p in _params}
assert not _opt.state
assert not {id(p) for p in _params} & {id(p) for p in model.parameters()}
_probe.close()
QUALIFICATION = {'passed_before_optimizer_step': True, 'disabled_max_abs_logit': _disabled_diff,
                 'zero_gamma_max_abs_logit': _zero_diff, 'near_init_max_abs_logit': _near_diff,
                 'injection_idx': list(SITES), 'train_sha256': TRAIN_SHA,
                 'eval_manifest_sha256': sha(WORK / 'evaluation-streams.json'),
                 'cache_sha256': CACHE_SHA, 'mapping_sha256': ADDR_SHA,
                 'lookup_max_abs_difference': _d, 'optimizer_tensor_count': len(_params),
                 'dtype': str(ACTIVE_DTYPE), 'target_revision': '851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a'}
QUALIFICATION['execution'] = EXECUTION
QUALIFICATION['execution_parity_sha256'] = sha(WORK/'execution-parity.json')
(WORK / 'qualification.json').write_text(json.dumps(QUALIFICATION, indent=2))
del reader, _probe, _params, _opt, _stock, _disabled, _zero, _near, _qual_ids, _qual_mem
gc.collect(); torch.cuda.empty_cache()
print('QUALIFICATION_PASSED', json.dumps(QUALIFICATION), flush=True)


In [ ]:
_bench_hits = list(Path('/kaggle/input').rglob('benchmark-4b.json'))
assert len(_bench_hits) == 1, 'Persist qualification before real training'
BENCHMARK = json.loads(_bench_hits[0].read_text())
assert BENCHMARK['configuration_qualified'] and BENCHMARK['passed_before_optimizer_step']
assert BENCHMARK['execution'] == EXECUTION
assert BENCHMARK['target_revision'] == trev
assert BENCHMARK['injection_idx'] == list(SITES)
assert BENCHMARK['train_sha256'] == TRAIN_SHA
assert BENCHMARK['lookup_max_abs_difference'] == 0


In [ ]:
# Cell 11 — training + eval core (val PASSED IN, never built here; threshold-crossing ckpts)
import math, time, json
import torch, torch.nn.functional as F

# addresses() lives in Cell 3 (single canonical tokens->PLE path)
def eval_loss(inj, store, val, use_reader=True):
    tot=ct=0; gates=[]
    with torch.inference_mode():
        for blk in val:
            blk=blk.unsqueeze(0).cuda()
            # FIX: ngram_indices BEFORE lookup — never raw token ids
            inj.set_memory(store.lookup(addresses(blk.cpu())).to('cuda') if use_reader else None)
            lg=model(input_ids=blk,use_cache=False).logits
            tot+=F.cross_entropy(lg[:,:-1].float().reshape(-1,lg.shape[-1]), blk[:,1:].reshape(-1), reduction='sum').item()
            ct+=blk[:,1:].numel()
            if use_reader:
                for r in inj.readers.values(): gates.append(r.last_gate.float().cpu().reshape(-1))
    l=tot/ct; g=torch.cat(gates) if gates else None
    import math as _m
    return {'loss':l,'ppl':_m.exp(min(l,20)),'gate_mean':g.mean().item() if g is not None else 0,'gate_std':g.std(correction=0).item() if g is not None else 0}

def train_reader(layers, branches=1, max_tokens=None, store=None, val=None, tag='run', ckpts=None, val_full=None, baselines=None, full_at=None, resume=None, cont_peak=None, cont_warm=200):
    # full_at: thresholds evaluated on val_full (default {max(ckpts)} = legacy behavior).
    # resume: ckpt dict from a previous train_reader save (reader+optimizer+scheduler+seen).
    # Every ckpt saves optimizer/scheduler/rng state so any leg can resume or persist.
    # val schedule: fast val below the final threshold, full val at the final threshold.
    # baselines: {'fast':..,'full':..} computed once and shared by all placements (None=recompute, smoke path).
    import random
    from torch.optim import AdamW
    assert val is not None, 'pass a frozen artifact from load_validation(); training must never build validation'
    max_tokens=max_tokens or C.SMOKE_TOKENS; ckpts=sorted(ckpts or [c for c in C.CKPTS if c<=max_tokens] or [max_tokens])
    random.seed(C.SEED); torch.manual_seed(C.SEED)
    inj=ReaderInjection(model,list(layers),C.MEM_DIM,C.HIDDEN,branches,C.GAMMA_INIT).to('cuda')  # FP32 reader+AdamW
    tr=[p for _,p in inj.named_parameters() if p.requires_grad]
    assert tr and not any(p.requires_grad for p in model.parameters())
    init_state={n: p.detach().cpu().clone() for n, p in inj.named_parameters()}
    def _unorm(pred):
        _t=0.0
        for _n,_p in inj.named_parameters():
            if pred(_n): _t+=(_p.detach().float().cpu()-init_state[_n].float()).pow(2).sum().item()
        return math.sqrt(_t)
    n_steps=max(1,max_tokens//C.SEQ); warm=max(1,math.ceil(n_steps*C.WARMUP_FRAC))
    opt=AdamW(tr,lr=C.LR,weight_decay=C.WD)
    assert {id(p) for g in opt.param_groups for p in g['params']} == {id(p) for p in inj.parameters()}
    _seen0=0; _skip=0
    if resume is not None:
        inj.load_state_dict({k: v.to('cuda') for k, v in resume['reader'].items()})
        opt.load_state_dict(resume['optimizer'])
        for _st in opt.state.values():
            for _k2, _v2 in _st.items():
                if isinstance(_v2, torch.Tensor): _st[_k2]=_v2.to('cuda')
        _seen0=int(resume['seen']); _skip=_seen0//C.SEQ
    # Continuation-aware LR: fresh cosine decays to 0 at its horizon, so naive restore
    # of a 1M-horizon scheduler into a 5M-horizon run restarts near peak (2.8e-05 shock).
    # If cont_peak is set with resume, do NOT load the old scheduler; instead re-warm
    # 0->cont_peak over cont_warm steps from the resume point, then cosine to 0 at 5M.
    if resume is not None and cont_peak is not None:
        _S0=int(resume['seen'])//C.SEQ; _S1=-(-max_tokens//C.SEQ); _fac=float(cont_peak)/C.LR
        def _cont_lam(s, _S0=_S0, _S1=_S1, _fac=_fac, _w=int(cont_warm)):
            if s<_S0+_w: return _fac*max(0.0,(s-_S0+1))/max(1,_w)
            _t=min((s-_S0-_w)/max(1,_S1-_S0-_w),1.0)
            return _fac*0.5*(1+math.cos(math.pi*_t))
        sch=torch.optim.lr_scheduler.LambdaLR(opt, _cont_lam)
        sch.last_epoch=_skip  # align to global step; first resumed step uses loaded ~0 LR, then ramps
        print('continuation schedule: S0 %d S1 %d peak %g warm %d (old scheduler NOT restored)' % (_S0,_S1,float(cont_peak),int(cont_warm)), flush=True)
    else:
        sch=torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s+1)/warm if s<warm else 0.5*(1+math.cos(math.pi*min((s-warm)/max(1,n_steps-warm),1))))
        if resume is not None and 'scheduler' in resume: sch.load_state_dict(resume['scheduler'])
    if resume is not None and 'torch_rng' in resume:
        torch.set_rng_state(resume['torch_rng']); torch.cuda.set_rng_state_all(resume['cuda_rng'])
        random.setstate(resume['python_rng'])
    if resume is not None and 'init_state' in resume:
        init_state={n:p.clone() for n,p in resume['init_state'].items()}
    if baselines is None:
        base=eval_loss(inj,store,val,False); print('frozen baseline:',{k:round(v,5) if isinstance(v,float) else v for k,v in base.items()})
    else:
        base=None  # cached per-threshold baselines shared across placements
    chunks=FROZEN_TRAIN[:math.ceil(max_tokens/C.SEQ)]
    full_set=set(full_at) if full_at else {max(ckpts)}
    pending=[c for c in ckpts if c>_seen0]; res=[]; iloss=0; icount=0; t0=time.perf_counter()
    if resume is not None and _skip: chunks=chunks[_skip:]
    assert store.meta['train_start'] <= _seen0
    assert store.meta['train_end'] >= math.ceil(max_tokens/C.SEQ)*C.SEQ
    assert torch.equal(chunks, FROZEN_TRAIN[_skip:_skip+len(chunks)])
    assert tuple(layers) == SITES and branches == 1
    assert len(tr) == 8 and all(p.dtype == torch.float32 for p in tr)
    assert not set(tr) & set(model.parameters())
    if resume is None: assert not opt.state
    else:
        assert set(opt.state) == set(tr)
        assert resume['corpus_cursor'] == {'next_block':_skip, 'tokens':_seen0, 'stream_sha256':TRAIN_SHA}
    gpu_reset()
    print('frozen training stream SHA verified', TRAIN_SHA, flush=True)
    for step, _ids in enumerate(chunks, start=_skip+1):
        ids_t=_ids.unsqueeze(0)
        inj.set_memory(store.lookup(addresses(ids_t)).to('cuda'))
        opt.zero_grad(set_to_none=True)
        lg=model(input_ids=ids_t.cuda(),use_cache=False).logits
        loss=F.cross_entropy(lg[:,:-1].float().reshape(-1,lg.shape[-1]), ids_t.cuda()[:,1:].reshape(-1))
        if not torch.isfinite(loss): raise RuntimeError('non-finite loss: abort this run')
        loss.backward()
        assert not any(p.grad is not None for p in model.parameters()), 'backbone grad leak'
        gn=torch.nn.utils.clip_grad_norm_(inj.parameters(),1.0,error_if_nonfinite=True).item(); opt.step(); sch.step()
        seen=step*C.SEQ; iloss+=loss.item()*(C.SEQ-1); icount+=C.SEQ-1
        if step % 1000 == 0:
            print('TRAIN_PROGRESS', seen, round((seen-_seen0)/(time.perf_counter()-t0),1), flush=True)
        if step % 100 == 0:
            print('TRAIN_PROGRESS', json.dumps({'step':step, 'tokens':seen, 'loss':loss.item(),
                  'lr':opt.param_groups[0]['lr'], 'tok_s':(seen-_seen0)/(time.perf_counter()-t0),
                  'gpu_peaks_gib':gpu_peaks(), 'host_rss_mib':rss_mb()}), flush=True)
        while pending and seen>=pending[0]:  # threshold-crossing, not exact multiples
            th=pending.pop(0)
            _vv=val_full if (val_full is not None and th in full_set) else val
            assert BACKBONE_VERSIONS == {n:p._version for n,p in model.named_parameters()}
            assert PLE_STATS == {str(p):(p.stat().st_size,p.stat().st_mtime_ns) for p in _cstore.part_paths.values()}
            assert set(opt.state) == set(tr)
            pre_eval_save_reader(inj,seen,th)
            v=eval_loss(inj,store,_vv,True)
            _base=(baselines['full' if _vv is val_full else 'fast'] if isinstance(baselines,dict) else base)
            gam={n:p.item() for n,p in inj.named_parameters() if n.endswith('gamma')}
            r={'tokens':seen,'threshold':th,'val_set':('full' if _vv is val_full else 'fast'),'train_loss':iloss/icount,'val_loss':v['loss'],'dval':v['loss']-_base['loss'],
               'ppl':v['ppl'],'lr':float(opt.param_groups[0]['lr']),'gamma':gam,'gate':(v['gate_mean'],v['gate_std']),'gnorm':gn,
               'peak_GiB':round(max(gpu_peaks()),3), 'gpu_peaks_gib':gpu_peaks(),'tok_s':round((seen-_seen0)/(time.perf_counter()-t0),1),
               'w_k_update':_unorm(lambda n: '.keys.' in n and n.endswith('.weight')),'w_v_update':_unorm(lambda n: n.endswith('.value.weight')),'beta_update':_unorm(lambda n: n.endswith('beta')),'rss_MiB':round(rss_mb(),1),'wall_s':round(time.perf_counter()-t0,1)}
            res.append(r); print(json.dumps(r,indent=1))
            import copy as _copy
            _osd=_copy.deepcopy(opt.state_dict())  # state_dict aliases live opt tensors: never .cpu() them in place
            _ckpt={'reader':{k:v.cpu() for k,v in inj.state_dict().items()},'cfg':(tuple(layers),branches),
                    'optimizer':_osd,
                    'scheduler':sch.state_dict(),'seen':seen,'iloss':iloss,'icount':icount,
                    'init_state':{n:p.cpu() for n,p in init_state.items()},
                    'torch_rng':torch.get_rng_state(),'cuda_rng':torch.cuda.get_rng_state_all(),
                    'python_rng':random.getstate(),'optimizer_step':step,
                    'corpus_cursor': {'next_block':step, 'tokens':seen, 'stream_sha256':TRAIN_SHA},
                    'provenance':RUN_META}
            for _st in _ckpt['optimizer']['state'].values():
                for _k2,_v2 in _st.items():
                    if isinstance(_v2,torch.Tensor): _st[_k2]=_v2.cpu()
            atomic_checkpoint(_ckpt, WORK / ('reader-%s-%d.pt' % (tag, seen)))
            iloss=0; icount=0
    inj.close(); return {'baseline':base,'baselines':baselines,'checkpoints':res}


In [ ]:
import copy, math, os, random

def same_state(a, b):
    if isinstance(a, torch.Tensor): return isinstance(b, torch.Tensor) and torch.equal(a.cpu(), b.cpu())
    if isinstance(a, dict): return a.keys() == b.keys() and all(same_state(a[k], b[k]) for k in a)
    if isinstance(a, (list, tuple)): return type(a) is type(b) and len(a) == len(b) and all(same_state(x, y) for x, y in zip(a, b))
    return a == b

def atomic_checkpoint(state, path):
    tmp = path.with_suffix(path.suffix + '.tmp')
    with tmp.open('wb') as f:
        torch.save(state, f)
        f.flush(); os.fsync(f.fileno())
    test = torch.load(tmp, map_location='cpu', weights_only=False)
    assert same_state(test, state)
    os.replace(tmp, path)
    fd = os.open(path.parent, os.O_RDONLY)
    try: os.fsync(fd)
    finally: os.close(fd)
    print('CHECKPOINT', path.name, sha(path), flush=True)

def pre_eval_save_reader(inj, seen, threshold):
    path = WORK / ('reader-%d.safetensors' % seen)
    tmp = path.with_suffix('.safetensors.tmp')
    state = {k: v.detach().cpu() for k, v in inj.state_dict().items()}
    save_file(state, str(tmp))
    with tmp.open('rb+') as f: f.flush(); os.fsync(f.fileno())
    check = load_file(str(tmp), device='cpu')
    assert set(check) == set(state)
    assert all(torch.equal(check[k], v) for k, v in state.items())
    os.replace(tmp, path)
    fd = os.open(path.parent, os.O_RDONLY)
    try: os.fsync(fd)
    finally: os.close(fd)
    print('READER', threshold, seen, sha(path), flush=True)

RUN_META = {'target_revision': '851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a', 'tokenizer_sha256': '5f9e4d4901a92b997e463c1f46055088b6cca5ca61a6522d1b9f64c4bb81cb42',
            'ple_revision': '236dfdf285828023ca3bcd3f37366c58a3469b13', 'train_stream_sha256': TRAIN_SHA,
            'train_manifest_sha256': TRAIN_MANIFEST_SHA,
            'eval_streams_sha256': sha(WORK / 'evaluation-streams.json'),
            'compact_cache_sha256': CACHE_SHA, 'address_mapping_sha256': ADDR_SHA,
            'injection_idx': [3, 11], 'human_layers': [4, 12],
            'reader_branches': 1, 'hidden': 2560,
            'lr': C.LR, 'weight_decay': C.WD, 'gamma_init': C.GAMMA_INIT,
            'schedule': {'mode': 'continuation', 'peak': 1.5e-5,
                         'warm_steps': 200, 'horizon_tokens': 10000000}}
RUN_META.update({'model_manifest_sha256': 'f8e8c8d5826d4abfa19bea82f85f0411e35684deb4fa8ed047e0c90ba94dc9c3', 'protocol_sha256': 'b78bab6c7a0a2b67a694946b0cac0bea0b04f0373b7418a0934366714ea3d32a', 'ple_manifest_sha256': 'bea49feaf18f97b7521c3c74773d07b9cdfd34453b6b420ee93d48858bd0c496', 'tokenizer_revision': '851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a', 'relative_depth': [0.125, 0.375], 'execution': {'strategy': 'single-T4-checkpointed', 'gpu': 0, 'decoder_gpu0': [0, 31], 'reader_devices': {'3': 0, '11': 0}, 'embedding_and_tied_head_gpu': 0, 'norm_gpu': 0, 'use_reentrant': False, 'preserve_rng_state': True, 'checkpoint_scope': 'decoder forward bodies only; module injection hooks execute once', 'reason': 'User requested single-GPU execution after ordinary single-T4 backward OOM', 'stock_logit_atol': 0.002, 'reader_gradient_atol': 1e-06, 'reader_gradient_rtol': 0.0001, 'reader_and_arbitration_math_unchanged': True, 'cuda_visible_devices': '0'}, 'execution_sha256': '7dad80a97996f8eb15ee80270b12a6771f5492689334f24b6d626e5f8b0c624e', 'execution_code_sha256': '54841a09e7a8ed5c815cd9217974d5fee13dcb467779d1273869787357c0453c'})
assert not any(p.requires_grad for p in model.parameters())
assert _cmp.meta['train_tokens_sha256'] == hashlib.sha256(FROZEN_TRAIN[5000192//512:10000384//512].numpy().astype('<u4').tobytes()).hexdigest()
PLE_STATS = {str(p): (p.stat().st_size, p.stat().st_mtime_ns) for p in _cstore.part_paths.values()}
summary_hits = list(Path('/kaggle/input').rglob('reader-5m-summary.json'))
assert len(summary_hits) == 1
previous = json.loads(summary_hits[0].read_text())
resume_path = summary_hits[0].parent / 'reader-real-5m-5000192.pt'
assert sha(resume_path) == previous['checkpoint_sha256']
resume = torch.load(resume_path, map_location='cpu', weights_only=False)
assert resume['seen'] == 5000192
assert resume['provenance']['train_stream_sha256'] == TRAIN_SHA
assert resume['cfg'] == ((3, 11), 1)
assert len(resume['optimizer']['state']) == 8
base = ReaderInjection(model, [3, 11], C.MEM_DIM, C.HIDDEN, 1, 0.0).to('cuda')
stock_fast = eval_loss(base, None, val_fast, False)
base.close()
baselines = {'fast': stock_fast}
out = train_reader([3, 11], 1, 10000000, _cmp, val_fast, 'real-10m',
                   ckpts=[10000000], baselines=baselines,
                   resume=resume, cont_peak=1.5e-5, cont_warm=200)
seen = out['checkpoints'][-1]['tokens']
assert seen == 10000384
ck = WORK / ('reader-real-10m-%d.pt' % seen)
verified = torch.load(ck, map_location='cpu', weights_only=False)
assert verified['seen'] == seen and verified['provenance']['train_stream_sha256'] == TRAIN_SHA
assert PLE_STATS == {str(p): (p.stat().st_size, p.stat().st_mtime_ns) for p in _cstore.part_paths.values()}
summary = {'stock_fast': stock_fast,
           'reader': out['checkpoints'][-1],
           'reader_sha256': sha(WORK / ('reader-%d.safetensors' % seen)),
           'checkpoint_sha256': sha(ck), 'train_sha256': TRAIN_SHA,
           'cache_sha256': CACHE_SHA, 'address_sha256': ADDR_SHA}
(WORK / 'reader-10m-summary.json').write_text(json.dumps(summary, indent=2))
print('READER_10M_COMPLETE', json.dumps(summary), flush=True)

assert BACKBONE_VERSIONS == {n:p._version for n,p in model.named_parameters()}
